# Notebook 26 — Hugging Face Inference APIs and Open-Model Selection

    ## Learning objectives

    - Move from pipeline to forward passes and hosted endpoints
- Select models using measured compatibility and constraints
- Control device maps, dtypes, batching, trust, and revisions

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'huggingface-hub>=0.30,<1', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 26.1 Select an artifact, not a leaderboard row

Define task, languages, modalities, latency, throughput, memory, context, license, privacy, tool/structure needs, and hardware before choosing a model. Distinguish base, instruct, reasoning, dense, and MoE variants; total and active parameters differ. Inspect model and dataset cards, configuration, tokenizer, chat template, architecture support, quantized derivatives, and immutable revisions. Leaderboards suggest candidates but may use different prompts, templates, contamination, and hardware. Run a task-specific baseline and record exclusions.


In [ ]:
requirements={"license":"commercial-compatible","memory_gib":16,"tools":True,"min_context":8192}; candidates=[{"id":"a","memory_gib":8,"tools":True,"context":32768},{"id":"b","memory_gib":20,"tools":True,"context":65536}]
print([c for c in candidates if c["memory_gib"]<=requirements["memory_gib"] and c["tools"] and c["context"]>=requirements["min_context"]])


## 26.2 Abstraction ladder

`pipeline` provides task preprocessing and postprocessing for fast baselines. Auto tokenizer/processor plus AutoModel exposes batching and generation. Direct forward passes expose logits, loss, hidden states, and cache behavior. `InferenceClient` targets remote providers; dedicated endpoints and local servers change operational ownership. Choose the highest abstraction that exposes required controls, then inspect what it adds. Do not mix pipeline-returned full text with newly generated tokens or assume model-specific output schemas are uniform.


In [ ]:
try:
 from transformers import pipeline
 print("pipeline is the baseline API; guarded to avoid an implicit model download")
except ImportError: print("run setup")


## 26.3 Loading and memory

Pin model and tokenizer commits together. Review `trust_remote_code` and custom repositories before execution. Choose dtype supported by hardware, use device maps deliberately, and understand offload latency. `low_cpu_mem_usage`, Safetensors, sharded checkpoints, quantization, and attention implementations affect load and residency. Set pad token and padding side based on task. Measure peak memory after warmup, not just weight size; KV cache and workspaces grow with live tokens.


In [ ]:
import torch
for dtype in [torch.float32,torch.float16,torch.bfloat16]: print(dtype,torch.finfo(dtype).bits)


## 26.4 Compatibility and deployment

Test chat rendering, greedy and sampled generation, stopping, streaming, structured output, tools, logprobs, batching, cancellation, and error behavior on the exact backend. A model loading in Transformers does not guarantee optimized vLLM or SGLang support. Keep a narrow internal request contract and engine conformance tests. Compare local inference, Hugging Face providers, managed endpoints, Ollama, and vLLM by data boundary, cold start, concurrency, cost, and operations. Store raw predictions and release a model-selection decision record.


In [ ]:
contract={"chat":True,"stream":True,"tools":True,"json_schema":True,"cancel":True}; observed={"chat":True,"stream":True,"tools":False,"json_schema":True,"cancel":False}
print("gaps",[k for k,v in contract.items() if v and not observed.get(k)])


## Reference workflow and evidence standard

Treat the notebook as an experiment, not a recipe. State the question, freeze inputs and
success criteria, establish the simplest baseline, change one material factor, and retain raw
outputs needed to diagnose failures. Record model, tokenizer, template, data and code revisions;
hardware and dtype; random seeds; generation or optimization configuration; token counts;
latency and memory; and results by meaningful slice. A demonstration that runs is evidence of
plumbing, not evidence of general capability.

Test boundaries as well as the happy path: empty and maximum-length inputs, malformed records,
multilingual or code text, unavailable dependencies, cancellation, and adversarial content.
Keep credentials in environment or Colab Secrets and never serialize them with artifacts. Pin
remote revisions, review licenses and custom code, validate saved artifacts in a fresh process,
and prefer deterministic validators wherever outputs can be checked mechanically.

Before applying the technique, compare it with prompting, retrieval, a smaller model, or no
model. Report quality together with compute, storage, latency, and operational complexity. Use
held-out data and paired comparisons, disclose uncertainty and negative results, and define a
rollback path. These practices connect low-level understanding to reliable application work.

A useful completion checklist asks four separate questions. Is the mathematical contract clear
enough to predict shapes, masks, reductions, and failure cases? Does the implementation reproduce
a tiny hand-worked or deterministic reference? Does the measured result survive a held-out set,
relevant slices, and an ablation against a simpler baseline? Can another person reload the exact
artifacts and reconstruct the claim from the manifest? Passing only the first two establishes a
tutorial demonstration; passing all four supports an engineering decision. When a result fails,
preserve the counterexample and update the test suite before changing the implementation.

Finally, separate correctness, capability, efficiency, and safety conclusions. A correct
implementation may have weak capability; a capable prototype may be too costly or unsafe to
deploy. Name the population to which each conclusion applies and avoid converting a single
metric into a universal ranking. Track assumptions beside results, especially tokenizer and
template compatibility, data rights, access-control boundaries, and hardware-specific behavior.
Leave exercises with an executable acceptance criterion, a baseline result, and a short written
interpretation. That combination turns exploratory code into cumulative course evidence that can
be revisited when libraries, model families, or deployment engines change.


## Exercises

    1. Build a model-selection scorecard.
2. Compare pipeline and direct generate outputs.
3. Write a backend conformance suite.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
